In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import chi2_contingency
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.utils import get_column_letter


INPUT_FILE = "CuratedDataset.xlsx"
EXCEL_OUT = "Correlation_Matrices.xlsx"

df = pd.read_excel(INPUT_FILE)

numerical_features = [
    "OMR", "SSA",
    "AMR", "BR", "CR",
    "Molarity", "PW", "CD",
    "Capacitance",
]
categorical_features = [
    "Monomer", "Dopant", "Template", "Oxidant", "Electrolyteclass",
    "ACN", "Configuration",
]
all_features = numerical_features + categorical_features


for c in ["ACN", "Configuration"]:
    df[c] = df[c].astype("Int64").astype(str)

data_num = df[numerical_features].copy()
data_all = df[all_features].copy()

corr_pearson  = data_num.corr(method="pearson")
corr_spearman = data_num.corr(method="spearman")
corr_kendall  = data_num.corr(method="kendall")


def cramers_v(x, y):
    """Bias-corrected Cramer's V for two categorical Series."""
    s = pd.concat([x, y], axis=1).dropna()
    if len(s) < 2:
        return np.nan
    table = pd.crosstab(s.iloc[:, 0], s.iloc[:, 1])
    if table.shape[0] < 2 or table.shape[1] < 2:
        return 0.0
    chi2, _, _, _ = chi2_contingency(table)
    n = table.values.sum()
    phi2 = chi2 / n
    r, k = table.shape
    phi2_corr = max(0.0, phi2 - (r - 1) * (k - 1) / (n - 1))
    r_corr = r - (r - 1) ** 2 / (n - 1)
    k_corr = k - (k - 1) ** 2 / (n - 1)
    denom = min(r_corr - 1, k_corr - 1)
    if denom <= 0:
        return 0.0
    return float(np.sqrt(phi2_corr / denom))


def correlation_ratio(num, cat):
    """Correlation ratio eta = sqrt(SS_between / SS_total)."""
    s = pd.concat([num, cat], axis=1).dropna()
    if len(s) < 2:
        return np.nan
    y = s.iloc[:, 0].astype(float).values
    g = s.iloc[:, 1].astype(str).values
    overall_mean = y.mean()
    ss_total = ((y - overall_mean) ** 2).sum()
    if ss_total == 0:
        return 0.0
    ss_between = 0.0
    for level in np.unique(g):
        yk = y[g == level]
        ss_between += len(yk) * (yk.mean() - overall_mean) ** 2
    return float(np.sqrt(ss_between / ss_total))


def pairwise_association(a, b, type_a, type_b):
    if type_a == "num" and type_b == "num":
        s = pd.concat([a, b], axis=1).dropna()
        if len(s) < 2:
            return np.nan
        return float(abs(s.iloc[:, 0].corr(s.iloc[:, 1], method="pearson")))
    if type_a == "cat" and type_b == "cat":
        return cramers_v(a, b)
    if type_a == "num" and type_b == "cat":
        return correlation_ratio(a, b)
    return correlation_ratio(b, a)


types = {f: "num" for f in numerical_features}
types.update({f: "cat" for f in categorical_features})

mixed = pd.DataFrame(index=all_features, columns=all_features, dtype=float)
for fa in all_features:
    for fb in all_features:
        if fa == fb:
            mixed.loc[fa, fb] = 1.0
        else:
            mixed.loc[fa, fb] = pairwise_association(
                data_all[fa], data_all[fb], types[fa], types[fb]
            )

def plot_signed_heatmap(corr, title, out_png):
    """Diverging blue-white-red colormap for signed [-1, 1] coefficients."""
    fig, ax = plt.subplots(figsize=(8.5, 7))
    cmap = LinearSegmentedColormap.from_list(
        "bwr_soft",
        ["#2166AC", "#67A9CF", "#F7F7F7", "#EF8A62", "#B2182B"],
        N=256,
    )
    im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1, aspect="equal")
    n = len(corr.columns)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=10)
    ax.set_yticklabels(corr.columns, fontsize=10)
    for i in range(n):
        for j in range(n):
            v = corr.values[i, j]
            color = "white" if abs(v) > 0.55 else "black"
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=8, color=color)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Correlation coefficient", fontsize=11)
    cbar.ax.tick_params(labelsize=9)
    ax.set_title(title, fontsize=12, pad=12)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_unsigned_heatmap(mat, title, out_png):
    """Sequential white-to-red colormap for [0, 1] association strength."""
    fig, ax = plt.subplots(figsize=(9.2, 8))
    cmap = LinearSegmentedColormap.from_list(
        "wr_soft",
        ["#FFFFFF", "#FDDBC7", "#EF8A62", "#B2182B", "#67001F"],
        N=256,
    )
    vals = mat.astype(float).values
    im = ax.imshow(vals, cmap=cmap, vmin=0, vmax=1, aspect="equal")
    n = len(mat.columns)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(mat.columns, rotation=45, ha="right", fontsize=10)
    ax.set_yticklabels(mat.columns, fontsize=10)
    for i in range(n):
        for j in range(n):
            v = vals[i, j]
            color = "white" if v > 0.55 else "black"
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=8, color=color)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Association strength", fontsize=11)
    cbar.ax.tick_params(labelsize=9)
    ax.set_title(title, fontsize=12, pad=12)
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close(fig)


plot_signed_heatmap(corr_pearson,  "Pearson correlation matrix",
                    "S_Pearson_correlation.png")
plot_signed_heatmap(corr_spearman, "Spearman correlation matrix",
                    "S_Spearman_correlation.png")
plot_signed_heatmap(corr_kendall,  "Kendall correlation matrix",
                    "S_Kendall_correlation.png")
plot_unsigned_heatmap(
    mixed,
    "Mixed-type association matrix\n"
    "(|Pearson| | Cramer's V | correlation ratio eta)",
    "S_Mixed_association.png",
)

with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl") as writer:
    corr_pearson.to_excel(writer,  sheet_name="Pearson")
    corr_spearman.to_excel(writer, sheet_name="Spearman")
    corr_kendall.to_excel(writer,  sheet_name="Kendall")
    mixed.astype(float).to_excel(writer, sheet_name="Mixed")

wb = load_workbook(EXCEL_OUT)
header_font = Font(name="Arial", bold=True, size=11)
body_font   = Font(name="Arial", size=10)
header_fill = PatternFill("solid", start_color="D9E1F2")
center      = Alignment(horizontal="center", vertical="center")

signed_scale = ColorScaleRule(
    start_type="num", start_value=-1, start_color="2166AC",
    mid_type="num",   mid_value=0,    mid_color="F7F7F7",
    end_type="num",   end_value=1,    end_color="B2182B",
)
unsigned_scale = ColorScaleRule(
    start_type="num", start_value=0,   start_color="FFFFFF",
    mid_type="num",   mid_value=0.5,   mid_color="EF8A62",
    end_type="num",   end_value=1,     end_color="67001F",
)

sheet_specs = [
    ("Pearson",  numerical_features, signed_scale),
    ("Spearman", numerical_features, signed_scale),
    ("Kendall",  numerical_features, signed_scale),
    ("Mixed",    all_features,       unsigned_scale),
]

for sheet_name, feats, rule in sheet_specs:
    ws = wb[sheet_name]
    n_rows = len(feats) + 1
    n_cols = len(feats) + 1

    for col_idx in range(1, n_cols + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = center

    for row_idx in range(2, n_rows + 1):
        idx_cell = ws.cell(row=row_idx, column=1)
        idx_cell.font = header_font
        idx_cell.fill = header_fill
        for col_idx in range(2, n_cols + 1):
            cell = ws.cell(row=row_idx, column=col_idx)
            cell.font = body_font
            cell.alignment = center
            cell.number_format = "0.000"

    data_range = (
        f"{get_column_letter(2)}2:"
        f"{get_column_letter(n_cols)}{n_rows}"
    )
    ws.conditional_formatting.add(data_range, rule)

    ws.column_dimensions["A"].width = 22
    for col_idx in range(2, n_cols + 1):
        ws.column_dimensions[get_column_letter(col_idx)].width = 14

wb.save(EXCEL_OUT)

num_summary = pd.DataFrame({
    "Pearson":  corr_pearson["Capacitance"],
    "Spearman": corr_spearman["Capacitance"],
    "Kendall":  corr_kendall["Capacitance"],
}).drop("Capacitance")

print("\nClassical correlation with Capacitance (numerical features only):")
print(num_summary.round(3).to_string())

cat_summary = mixed.loc[categorical_features, "Capacitance"].astype(float)
print("\nCorrelation ratio eta with Capacitance (categorical features):")
print(cat_summary.round(3).to_string())

print(f"\nExcel file written: {EXCEL_OUT}")
print("Heatmaps written: Pearson correlation.png, "
      "Spearman correlation.png, Kendall correlation.png, "
      "Mixed association.png")


Classical correlation with Capacitance (numerical features only):
          Pearson  Spearman  Kendall
OMR         0.014     0.040    0.032
SSA         0.236     0.249    0.169
AMR         0.174     0.155    0.112
BR         -0.198    -0.170   -0.130
CR         -0.094    -0.072   -0.056
Molarity   -0.091    -0.101   -0.079
PW         -0.165    -0.182   -0.140
CD         -0.168    -0.308   -0.222

Correlation ratio eta with Capacitance (categorical features):
Monomer             0.236
Dopant              0.192
Template            0.348
Oxidant             0.201
Electrolyteclass    0.270
ACN                 0.227
Configuration       0.063

Excel file written: Correlation_Matrices.xlsx
Heatmaps written: Pearson correlation.png, Spearman correlation.png, Kendall correlation.png, Mixed association.png
